# grid-rbd analytic gradients — first & second order

GRiD emits **analytic** dynamics derivatives — no autodiff tape, no finite-difference at runtime. This notebook exercises them on the numpy backend and validates each against a finite-difference of the corresponding forward map (hermetic — one library, batched over axis 0):

- **First order:** `inverse_dynamics_gradient(q,qd,qdd)` = `∂c/∂(q,qd)`; `forward_dynamics_gradient(q,qd,u)` = `∂qdd/∂(q,qd)`.
- **Second order:** `idsva_so(q,qd,qdd)` and `fdsva_so(q,qd,u)`, each four `(B,NV,NV,NV)` tensors — validated against a finite-diff of the first-order gradient.

**Setup:** a CUDA GPU + `nvcc` on PATH, and `grid_rbd` installed editable from this repo — `pip install -e python/` from the repo root (see [`notebooks/README.md`](README.md)). iiwa14 caches in seconds.

In [ ]:
import numpy as np
from pathlib import Path
import grid_rbd

URDF = next(p / 'robot_assets' / 'iiwa14.urdf' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'robot_assets' / 'iiwa14.urdf').exists())
assert URDF.exists(), URDF
np.random.seed(0)

## 1. Register the robot

In [ ]:
h = grid_rbd.register_robot('iiwa14_grad', urdf_path=str(URDF),
                            floating_base=False, max_batch_size=64)
NJ, NV = h.num_joints, h.num_vel
print(h, '  NJ =', NJ, ' NV =', NV)

## 2. Inverse-dynamics gradient `∂c/∂(q, qd)`

`inverse_dynamics_gradient(q, qd, qdd)` returns `(B, NJ, 2*NJ)` = `[∂c/∂q | ∂c/∂qd]`. Slice `[..., :NJ]` and `[..., NJ:]`. We validate both blocks against a central finite-difference of `inverse_dynamics`.

In [ ]:
q   = np.random.randn(1, NJ).astype(np.float32) * 0.3
qd  = np.random.randn(1, NJ).astype(np.float32) * 0.3
qdd = np.random.randn(1, NJ).astype(np.float32) * 0.3

G = h.inverse_dynamics_gradient(q, qd, qdd)   # (1, NJ, 2*NJ)
dc_dq, dc_dqd = G[..., :NJ], G[..., NJ:]
print('inverse_dynamics_gradient:', G.shape)

def fd_jac(fn, x, eps=1e-3):
    """Central finite-diff Jacobian of fn(x)->(NJ,) wrt x (1,NJ) -> (NJ,NJ)."""
    n = x.shape[1]
    J = np.zeros((NJ, n), dtype=np.float64)
    for j in range(n):
        dx = np.zeros_like(x); dx[0, j] = eps
        J[:, j] = (fn(x + dx).astype(np.float64)[0]
                   - fn(x - dx).astype(np.float64)[0]) / (2 * eps)
    return J

fd_dq  = fd_jac(lambda qq: h.inverse_dynamics(qq, qd, qdd), q)
fd_dqd = fd_jac(lambda vv: h.inverse_dynamics(q, vv, qdd), qd)
e_dq  = np.abs(dc_dq[0].astype(np.float64)  - fd_dq).max()
e_dqd = np.abs(dc_dqd[0].astype(np.float64) - fd_dqd).max()
print(f'∂c/∂q  max|err| = {e_dq:.2e}    ∂c/∂qd max|err| = {e_dqd:.2e}')
assert e_dq < 2e-2 and e_dqd < 2e-2, (e_dq, e_dqd)

## 3. Forward-dynamics gradient `∂qdd/∂(q, qd)`

`forward_dynamics_gradient(q, qd, u)` returns `(B, NJ, 2*NJ)` = `[∂qdd/∂q | ∂qdd/∂qd]`, validated against a finite-diff of `forward_dynamics`.

In [ ]:
u = np.random.randn(1, NJ).astype(np.float32) * 0.3
Gf = h.forward_dynamics_gradient(q, qd, u)     # (1, NJ, 2*NJ)
df_dq, df_dqd = Gf[..., :NJ], Gf[..., NJ:]
print('forward_dynamics_gradient:', Gf.shape)

fd_dq  = fd_jac(lambda qq: h.forward_dynamics(qq, qd, u), q)
fd_dqd = fd_jac(lambda vv: h.forward_dynamics(q, vv, u), qd)
e_dq  = np.abs(df_dq[0].astype(np.float64)  - fd_dq).max()
e_dqd = np.abs(df_dqd[0].astype(np.float64) - fd_dqd).max()
print(f'∂qdd/∂q max|err| = {e_dq:.2e}    ∂qdd/∂qd max|err| = {e_dqd:.2e}')
assert e_dq < 5e-2 and e_dqd < 5e-2, (e_dq, e_dqd)

## 4. Second-order dynamics `idsva_so` / `fdsva_so` vs `RBDReference`

The second-order kernels return four `(B, NV, NV, NV)` tensors each. Their internal index layout is an implementation detail, so rather than re-derive it with a finite-difference we validate **block-for-block against the verified `RBDReference` Python implementation** — the exact oracle the CUDA equivalence tests use. The codegen dispatcher picks body-frame (fixed base) / world-frame (floating); for fixed-base iiwa14 both surfaces use body-frame.

- `idsva_so(q, qd, qdd)` → `(d2tau_dq, d2tau_dqd, d2tau_cross, dM_dq)` = `RBDReference.idsva_so(q, qd, qdd)`.
- `fdsva_so(q, qd, u)` → `(daba_dqdq, daba_dvdq, daba_dvdv, daba_dtdq)` = `RBDReference.fdsva_so(q, qd, u)`.

In [ ]:
from URDFParser import URDFParser
from RBDReference import RBDReference
ref = RBDReference(URDFParser().parse(str(URDF), floating_base=False))

# float64 reference inputs (the kernels run float32).
q64, qd64, qdd64, u64 = (a[0].astype(np.float64) for a in (q, qd, qdd, u))

def _max_rel(cuda_blocks, ref_blocks, names):
    worst = 0.0
    for name, A, R in zip(names, cuda_blocks, ref_blocks):
        A = np.asarray(A[0], dtype=np.float64); R = np.asarray(R, dtype=np.float64)
        scale = max(1.0, float(np.max(np.abs(R))))
        e = float(np.max(np.abs(A - R))) / scale
        print(f'  {name:11s} shape={A.shape} max|err|/scale = {e:.2e}')
        worst = max(worst, e)
    return worst

idsva_cuda = h.idsva_so(q, qd, qdd)            # 4 x (1, NV, NV, NV)
idsva_ref  = ref.idsva_so(q64, qd64, qdd64, GRAVITY=-9.81)
print('idsva_so vs RBDReference:')
w = _max_rel(idsva_cuda, idsva_ref, ('d2tau_dq','d2tau_dqd','d2tau_cross','dM_dq'))
assert w < 1e-2, w   # float32 CUDA vs float64 reference

## 5. Second-order forward dynamics `fdsva_so`

Same oracle check for the second-order **forward** dynamics tensors.

In [ ]:
fdsva_cuda = h.fdsva_so(q, qd, u)             # 4 x (1, NV, NV, NV)
fdsva_ref  = ref.fdsva_so(q64, qd64, u64, GRAVITY=-9.81)
print('fdsva_so vs RBDReference:')
w = _max_rel(fdsva_cuda, fdsva_ref, ('daba_dqdq','daba_dvdq','daba_dvdv','daba_dtdq'))
assert w < 2e-2, w

Analytic first-order (`inverse_dynamics_gradient`, `forward_dynamics_gradient`, vs finite-difference) and second-order (`idsva_so`, `fdsva_so`, vs the `RBDReference` oracle) derivatives all validated. These are the kernels MPC/DDP solvers consume — exact, batched, no tape.